In [131]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os, sys
utils_path = os.path.abspath(os.path.join(os.getcwd(), '..', '2_data_analysis', 'utils'))
sys.path.append(utils_path)
import plot_style

C_AGR   = '#C0392B'
C_NOAGR = '#2C6E8A'

# **Spell Construction for Survival Analysis**

In this notebook, we will construct the spell dataset for the survival analysis, taking the <code>conflict_panel</code> dataset and adding the termination outcome and termination date for each conflict, that will determine the spell and outcome for the survival analysis. Then we will create the proxies to classify the conflicts into information asymmetry or commitment problems. Finally, we will aggregate the spell at quarterly level.

1. **Conflict Panel**
2. **Termination Outcome**
3. **Information Asymmetry**
4. **Commitment Problems**
5. **Quarterly Aggregation**

### Conflict Panel

In [132]:
df_panel = pd.read_csv('../../data/output/conflict_level/conflict_panel.csv', low_memory=False)

print(f"Total conflicts: {len(df_panel['conflict_id'].unique())}")
print(f"With agreement:   {df_panel.loc[df_panel['ever_agreement']==1]['conflict_id'].nunique()}")
print(f"\t agreement during conflict: {df_panel.loc[(df_panel['ever_agreement']==1) & (df_panel['first_agreement_date']>=df_panel['start_date_numeric']) & (df_panel['first_agreement_date']<=df_panel['end_date_numeric'])]['conflict_id'].nunique()}")
print(f"\t agreement before start_date: {df_panel.loc[(df_panel['ever_agreement']==1) & (df_panel['first_agreement_date']<df_panel['start_date_numeric'])]['conflict_id'].nunique()}")
print(f"\t agreement after end_date: {df_panel.loc[(df_panel['ever_agreement']==1) & (df_panel['first_agreement_date']>df_panel['end_date_numeric'])]['conflict_id'].nunique()}")
print(f"Without agreement: {df_panel.loc[df_panel['ever_agreement']==0]['conflict_id'].nunique()}")

Total conflicts: 201
With agreement:   104
	 agreement during conflict: 82
	 agreement before start_date: 11
	 agreement after end_date: 11
Without agreement: 97


### Termination outcome

Using the termination dataset, we will add the last termination outcome and termination date to the conflict panel for the cases that have a termination. This termination outcome will determine the spell of each conflict for the survival analysis, following these different outcomes:

- Military victory or actor ceases --> not at risk (event)
- Peace agreement or ceasefire --> signers 
- Low activity --> fade
- <code>termination_end_date = nan</code> --> censored (right-censored)

In [133]:
from functions.build_termination_outcome import build_termination_outcome

df_panel = build_termination_outcome(
    df_panel,
    "../../data/input/ucdp/UCDPConflictTerminationDataset_v4_2024_Conflict.csv"
)

df_panel[['conflict_id', 'year_mo', 'agree_timing', 'termination_outcome_label', 'cause_label', 'effective_end_date']]

,conflict_id,year_mo,agree_timing,termination_outcome_label,cause_label,effective_end_date
0,205,1989-01,never_signed,Low activity,fade,2018-10
1,205,1989-02,never_signed,Low activity,fade,2018-10
2,205,1989-03,never_signed,Low activity,fade,2018-10
3,205,1989-04,never_signed,Low activity,fade,2018-10
4,205,1989-05,never_signed,Low activity,fade,2018-10
...,...,...,...,...,...,...
86827,16379,2024-08,never_signed,ongoing,censored,2024-12
86828,16379,2024-09,never_signed,ongoing,censored,2024-12
86829,16379,2024-10,never_signed,ongoing,censored,2024-12
86830,16379,2024-11,never_signed,ongoing,censored,2024-12


In [134]:
df_panel.groupby(['cause_label', 'termination_outcome_label'], dropna=False)['conflict_id'].nunique().reset_index(name='n_conflicts').sort_values('n_conflicts', ascending=False)

,cause_label,termination_outcome_label,n_conflicts
0,censored,ongoing,39
1,fade,Low activity,31
8,first_agreement,ongoing,31
10,not at risk,Victory (govt),20
4,first_agreement,Low activity,19
5,first_agreement,Peace agreement,18
3,first_agreement,Ceasefire,16
6,first_agreement,Victory (govt),7
9,not at risk,Actor ceases to exist,7
11,not at risk,Victory (rebels),6


In [135]:
pd.crosstab(
    df_panel['termination_outcome_label'],
    df_panel['ever_agreement'],
    values=df_panel['conflict_id'],
    aggfunc=pd.Series.nunique,
    margins=True,
    margins_name='Total'
)

ever_agreement,0,1,Total
termination_outcome_label,,,
Actor ceases to exist,7.0,3.0,10
Ceasefire,NaN,16.0,16
Low activity,31.0,19.0,50
Peace agreement,NaN,18.0,18
Victory (govt),20.0,7.0,27
Victory (rebels),6.0,4.0,10
ongoing,39.0,31.0,70
Total,103.0,98.0,201


### Incompatibility type

Territorial conflicts have harder-to-divide stakes, which can be proxy of commitment problems.

In [136]:
#merge the incompatibility column from UCDP/PRIO
df_prio = pd.read_csv('../../data/input/ucdp/UcdpPrioConflict_v25_1.csv', low_memory=False)
df_panel = df_panel.merge(df_prio[['conflict_id', 'year', 'incompatibility', 'type_of_conflict']], left_on = ['conflict_id', 'year'], right_on = ['conflict_id', 'year'], how = 'left')
df_panel['incompatibility'] = df_panel.groupby('conflict_id')['incompatibility'].ffill().bfill().astype(int)
incompatibility_map = {1: "territorial", 2: "government", 3: "other"}
df_panel['incompatibility'] = df_panel['incompatibility'].map(incompatibility_map)
df_panel['territorial_incompatibility'] = (df_panel['incompatibility'] == 'territorial').astype(int)
type_of_conflict_map = {1: "extrasystemic", 2: "intrastate", 3: "interstate", 4: "internationalized_intrastate"}
df_panel['type_of_conflict'] = df_panel.groupby('conflict_id')['type_of_conflict'].ffill().bfill().map(type_of_conflict_map)
df_panel[['conflict_id', 'year_mo', 'incompatibility', 'type_of_conflict']]

,conflict_id,year_mo,incompatibility,type_of_conflict
0,205,1989-01,territorial,interstate
1,205,1989-02,territorial,interstate
2,205,1989-03,territorial,interstate
3,205,1989-04,territorial,interstate
4,205,1989-05,territorial,interstate
...,...,...,...,...
86827,16379,2024-08,territorial,interstate
86828,16379,2024-09,territorial,interstate
86829,16379,2024-10,territorial,interstate
86830,16379,2024-11,territorial,interstate


In [137]:
pd.crosstab(
    df_panel['territorial_incompatibility'],
    df_panel['ever_agreement'],
    values=df_panel['conflict_id'],
    aggfunc=pd.Series.nunique,
    margins=True,
    margins_name='Total'
)

ever_agreement,0,1,Total
territorial_incompatibility,,,
0,37,41,78
1,66,57,123
Total,103,98,201


In [138]:
pd.crosstab(
    df_panel['territorial_incompatibility'],
    df_panel['cause_label'],
    values=df_panel['conflict_id'],
    aggfunc=pd.Series.nunique,
    margins=True,
    margins_name='Total'
)

cause_label,censored,fade,first_agreement,not at risk,Total
territorial_incompatibility,,,,,
0,19,5,41,13,78
1,20,26,57,20,123
Total,39,31,98,33,201


## Information asymmetry proxies

---

Past fighting **before the onset** between the same parties reduces uncertainty about relative strength. `experience_dir` is the discounted count of prior dyadic conflicts (δ = 0.95/yr) between the exact pair (side_a, side_b), and `experience_gov` is the discounted count of prior conflicts between the side_a and other groups. The composite `experience_total` is a weighted sum of both, and we define `high_ia_bin = 1` when `experience_total == 0` at onset.

*The rationale: higher experience reduces uncertainty, which may reduce the information asymmetry.*

### Experience fighting

In [139]:
from functions.build_experience_compound import build_experience_expanded

df_panel = build_experience_expanded(
    df_panel,
    "../../data/input/ucdp/Dyadic_v25_1.csv",
    delta=0.95,
    weight_direct=1.0,
    weight_government=0.5,
)

df_panel[['conflict_id', 'year_mo', 'experience_total', 'exp_direct', 'exp_government']]

Experience computed for 201 conflicts:
  exp_direct = 0 in 133/201 conflicts (66%)
  experience_total = 0 in 77/201 conflicts (38%)
  (fewer zeros = more variation for IA proxy)


,conflict_id,year_mo,experience_total,exp_direct,exp_government
0,205,1989-01,12.164151,8.270352,7.787599
1,205,1989-02,12.164151,8.270352,7.787599
2,205,1989-03,12.164151,8.270352,7.787599
3,205,1989-04,12.164151,8.270352,7.787599
4,205,1989-05,12.164151,8.270352,7.787599
...,...,...,...,...,...
86827,16379,2024-08,7.250709,0.000000,14.501418
86828,16379,2024-09,7.250709,0.000000,14.501418
86829,16379,2024-10,7.250709,0.000000,14.501418
86830,16379,2024-11,7.250709,0.000000,14.501418


In [140]:
df_panel['prior_experience'] = (df_panel['experience_total'] > 0).astype(int)
pd.crosstab(
    df_panel['prior_experience'],
    df_panel['cause_label'],
    values=df_panel['conflict_id'],
    aggfunc=pd.Series.nunique,
    margins=True,
    margins_name='Total'
)

cause_label,censored,fade,first_agreement,not at risk,Total
prior_experience,,,,,
0,12,13,35,17,77
1,27,18,63,16,124
Total,39,31,98,33,201


## **Commitment proxies**

---

Onset / time-invariant proxies following Fearon (1995) and Feedback §3. Dynamic measures (factions, external support) are Exercise B only and excluded here.

### Ethnic fractionalization (Fearon 2003)

In [141]:
# Fearon & Laitin (2003) ethnic fractionalization via QoG dataset.
BASE = '../../data/input/fearon_ethnic_fractionalization/'

def load_fe(fname, varname):
    df = pd.read_csv(BASE + fname, index_col=0, sep=';', decimal=',')
    return df.groupby('ccodealp')[varname].first().reset_index()

fe_vars = {
    'fe_etfra':   'qogdata_08_06_2026_fe_etfra.csv',
    'fe_cultdiv': 'qogdata_08_06_2026_fe_cultdiv.csv',
}

# ── Merge into df_panel ────────────────────────────────────────────────────────
for varname, fname in fe_vars.items():
    static = load_fe(fname, varname)
    df_panel = df_panel.drop(columns=[varname, 'ccodealp'], errors='ignore')
    df_panel = df_panel.merge(static, left_on='isocode', right_on='ccodealp', how='left')
    df_panel = df_panel.drop(columns='ccodealp')

# ── Impute fe_etfra for missing countries ──────────────────────────────────────
# Build regional means from QoG countries with data
_fe_raw   = pd.read_csv(BASE + 'qogdata_08_06_2026_fe_etfra.csv', index_col=0, sep=';', decimal=',')
_reg_lkp  = df_panel[['isocode', 'region']].drop_duplicates()
_fe_u     = _fe_raw.groupby('ccodealp')['fe_etfra'].first().reset_index()
_reg_mean = (
    _fe_u.merge(_reg_lkp, left_on='ccodealp', right_on='isocode', how='left')
    .groupby('region')['fe_etfra'].mean().to_dict()
)

# Manual overrides for countries not in QoG by their own ISO
_val    = lambda c: _fe_raw[_fe_raw['ccodealp']==c]['fe_etfra'].iloc[0] \
                   if c in _fe_raw['ccodealp'].values else None
_manual = {k: v for k, v in {'SRB': _val('SCG') or _val('YUG'), 'SSD': _val('SDN')}.items()
           if v is not None}

# Impute at country level (fe_etfra is country-invariant → no need to apply row-wise)
_iso_lookup = (
    df_panel[['isocode', 'region', 'fe_etfra']]
    .drop_duplicates('isocode')
    .copy()
)
_iso_lookup['fe_etfra_imp'] = _iso_lookup.apply(
    lambda r: r['fe_etfra'] if pd.notna(r['fe_etfra'])
              else _manual.get(r['isocode'], _reg_mean.get(r['region'])),
    axis=1
)
df_panel = df_panel.drop(columns='fe_etfra_imp', errors='ignore')
df_panel = df_panel.merge(_iso_lookup[['isocode', 'fe_etfra_imp']], on='isocode', how='left')
df_panel['high_fe_etfra_bin'] = (df_panel['fe_etfra_imp'] > 0.5).astype(int)

print(f"Manual imputation cases: {_manual}")
print(f"fe_etfra NaN (before imputation): {df_panel['fe_etfra'].isna().sum()} rows  "
      f"({df_panel[df_panel['fe_etfra'].isna()]['conflict_id'].nunique()} conflicts)")
print(f"fe_etfra_imp NaN (after imputation): {df_panel['fe_etfra_imp'].isna().sum()}")
print(f"high_fe_etfra_bin > 0.5: {df_panel['high_fe_etfra_bin'].mean():.1%} of rows")

Manual imputation cases: {'SSD': np.float64(0.71)}
fe_etfra NaN (before imputation): 2592 rows  (6 conflicts)
fe_etfra_imp NaN (after imputation): 0
high_fe_etfra_bin > 0.5: 69.7% of rows


### UN Peacekeeping at onset — `pko_onset` (Geo-PKO v2.3)

UN peacekeeping presence at conflict onset is a proxy for prior international engagement and third-party enforcement capacity (Walter 1997). A pre-existing PKO signals that external actors were already involved before the conflict, which may lower commitment problems for rebel demobilization.

**Source:** Geocoded Peacekeeping Operations Dataset (Geo-PKO v2.3, Uppsala University), location × month panel, 1994–2024.

**Coverage limitation:** Geo-PKO starts in 1994. Of the 201 conflicts in the panel, **98 have onset before 1994** (panel runs from 1989). For all pre-1994 conflicts `pko_onset = 0` by construction — not because there was no PKO but because data is unavailable. 

In [142]:
from functions.build_onset_covariates import build_pko_onset

df_panel = build_pko_onset(
    df_panel,
    "../../data/input/ucdp/Geo_PKO_v.2.3_location_month.csv",
)

pko_onset = 1: 25  0: 176  (of which pre-1994 onset: 98)
pko_ever  = 1: 73  0: 128


### Defensive alliances at onset — `defpact_onset` / `defpact_majpow_onset` (ATOP)

| Variable | Definition | Source |
|---|---|---|
| `defpact_onset` | =1 if country had any active defensive alliance at conflict onset | ATOP State-Year via QoG (`atop_defensive`), 1946–2018 |
| `defpact_majpow_onset` | =1 if at least one defensive alliance partner was a major power (USA, UK, France, Russia, China) | ATOP v5.1 Dyad-Year (`atop5_1dy.csv`), 1815–2018 |

Both are 0 for countries within ATOP coverage with no qualifying alliance. `NaN` only for the 4 conflicts with onset after 2022 (Mali 2023-08, Ethiopia 2023-05, Yemen 2023-12, Somalia 2024-12) — ATOP was last updated Aug 2022, so absent country-years through 2022 reliably indicate no alliance. The QoG state-year is an unbalanced panel (only alliance-members appear). COW→ISO crosswalk built from QoG (`ccodecow` → `ccodealp`).

In [143]:
from functions.build_onset_covariates import build_defpact_onset

df_panel = build_defpact_onset(
    df_panel,
    "../../data/input/atop/qogdata_08_07_2026.csv",
    "../../data/input/atop/ATOP_5.1/atop5_1dy.csv",
)

defpact_onset:        1=112  0=85  NaN=4
defpact_majpow_onset: 1=39  0=158  NaN=4


In [152]:
pd.crosstab(
    df_panel['defpact_majpow_onset'].fillna(0).astype(int),
    df_panel['cause_label'],
    values=df_panel['conflict_id'],
    aggfunc=pd.Series.nunique,
    margins=True,
    margins_name='Total'
)

cause_label,censored,fade,first_agreement,not at risk,Total
defpact_majpow_onset,,,,,
0,35,26,76,25,162
1,4,5,22,8,39
Total,39,31,98,33,201


### Power-shift risk — `loot_onset_strict` / `loot_onset` (PRIO-GRID)

Lootable resources in the conflict zone proxy the *risk* of a future power shift (Powell 2006): if rebels can exploit resources, their capacity may grow over time, making commitment to an agreement fragile.

**Geographic footprint:** GED events in the first 90 days from `start_date` — the onset quarter, fixed before conflict dynamics could alter the spatial extent.

| Variable | Definition | Resource source |
|---|---|---|
| `loot_onset_strict` | =1 if any resource layer is 1 for any onset cell at the onset year | Yearly only (`_y`): deposits with known discovery year ≤ onset. No future-contamination risk. |
| `loot_onset` | =1 if `loot_onset_strict`=1 or any static layer is 1 for any onset cell | Yearly + Static (`_y` + `_s`): adds undated deposits (unknown discovery year — minor contamination caveat). |

**Source:** PRIO-GRID 2.0 — Static Variables + Yearly Variables 1989–2014 (grid.prio.org). Resource layers: `petroleum`, `diamsec`, `diamprim`, `gem`, `drug`, `goldplacer`, `goldvein`, `goldsurface`. Yearly data covers up to 2014; conflicts with onset after 2014 use the 2014 values (forward-fill assumption: resources persist once discovered).

In [145]:
from functions.build_onset_covariates import build_loot_onset

df_panel = build_loot_onset(
    df_panel,
    "../../data/input/ucdp/GEDEvent_v25_1.csv",
    "../../data/input/prio_grid/PRIO-GRID Static Variables - 2026-07-08.csv",
    "../../data/input/prio_grid/PRIO-GRID Yearly Variables for 1989-2014 - 2026-07-08.csv",
)

loot_onset_strict: 1= 97  0=104  NaN=0
loot_onset:        1=112  0= 89  NaN=0


In [146]:
pd.crosstab(
    df_panel['loot_onset'],
    df_panel['cause_label'],
    values=df_panel['conflict_id'],
    aggfunc=pd.Series.nunique,
    margins=True,
    margins_name='Total'
)

cause_label,censored,fade,first_agreement,not at risk,Total
loot_onset,,,,,
0,24,16,39,10,89
1,15,15,59,23,112
Total,39,31,98,33,201


### Other onset correlates — `bdist_onset` / `capdist_onset` (PRIO-GRID)

Structural correlates of weak state reach and conflict geography, derived from the same PRIO-GRID onset footprint.

| Variable | Definition | Source |
|---|---|---|
| `bdist_onset` | Mean distance (km) to nearest national border across onset cells | PRIO-GRID `bdist3`, yearly 1989–2014 |
| `capdist_onset` | Mean distance (km) to country capital across onset cells | PRIO-GRID `capdist`, yearly 1989–2014 |

Low `bdist_onset` → conflict near a border → likely cross-border sanctuary for rebels.  
High `capdist_onset` → conflict far from capital → weaker state administrative reach.

> **`mountains_onset` not yet implemented:** requires downloading `mountains_mean` from PRIO-GRID Static Variables (not included in current download).

In [147]:
from functions.build_onset_covariates import build_distance_onset

df_panel = build_distance_onset(
    df_panel,
    "../../data/input/ucdp/GEDEvent_v25_1.csv",
    "../../data/input/prio_grid/PRIO-GRID Yearly Variables for 1989-2014 - 2026-07-08.csv",
)

bdist_onset   — mean=63.9 km  NaN=0
capdist_onset — mean=574.2 km  NaN=0


### Rough terrain — `mountains_onset` (PRIO-GRID)  

| Variable | Definition | Source |  
|---|---|---|  
| `mountains_onset` | Mean share of mountainous terrain across onset-quarter GED cells | PRIO-GRID 2.0 Static (`mountains_mean`) |  

High values indicate rough terrain — associated with rebel sanctuary and longer conflicts (Fearon & Laitin 2003).  
`mountains_mean` is time-invariant (static file), so no year lookup is needed.  


In [148]:
from functions.build_onset_covariates import build_mountains_onset

df_panel = build_mountains_onset(
    df_panel,
    ged_path    = "../../data/input/ucdp/GEDEvent_v25_1.csv",
    prio_static_path = "../../data/input/prio_grid/PRIO-GRID Static Variables - 2026-07-08.csv",
)

mountains_onset — mean=0.377  NaN=0


## **Spell Dataset**

---

A **survival (spell) dataset** records each conflict's history as a sequence of monthly
periods until an event or exit occurs. Each row is one **conflict × month** observation.

### Spell boundaries

Each conflict enters the risk set at `start_date` (first observed GED event) and exits at
`effective_end_date` (the true spell endpoint, set in `build_termination_outcome`).

| `cause_label` | `agree_timing` | `effective_end_date` | Notes |
|---|---|---|---|
| `first_agreement` | `during_ged` | agreement month (`first_agreement_date` → YYYY-MM) | 93 PA-X signers |
| `first_agreement` | `during_ged` | `c_ependdate` (YYYY-MM) | 2 UCDP-reclassified with orig PA-X (274, 390) |
| `first_agreement` | `ucdp_only` | `c_ependdate` (YYYY-MM) | 3 UCDP-only signers (292, 11348, 14074) |
| `fade` / `not at risk` | — | `c_ependdate` if GED > term, else `end_date` | Competing exit |
| `censored` | — | `end_date` (last GED month, ≤ 2024-12) | Right-censored |



### Event indicator

`is_first_agreement = 1` on the **last observed month** of each signer (`ever_agreement = 1`).
Using "last month in spell" rather than `year_mo_numeric == first_agreement_date` because
5 UCDP-reclassified conflicts have `first_agreement_date` in absolute scale (from `_term_numeric`)
which never matches `year_mo_numeric` (relative scale, 1 = 1989-01).

### Key columns

| Column | Level | Description |
|---|---|---|
| `ever_agreement` | conflict | 1 if conflict reaches a peace agreement in the spell |
| `cause_label` | conflict | Exit type: `first_agreement` / `fade` / `not at risk` / `censored` |
| `is_first_agreement` | month | 1 on the agreement month (event indicator for the hazard model) |
| `effective_end_date` | conflict | Spell exit date (YYYY-MM) |
| `start_date` | conflict | Spell entry date (first GED event, YYYY-MM) |
| `agree_timing` | conflict | Diagnostic: `during_ged` / `ucdp_only` / `pre_ged` / `never_signed` |
| `termination_outcome_label` | conflict | UCDP episode outcome (diagnostic; `cause_label` is used in models) |

### Competing risks

The focal event is `first_agreement`. Non-signers exit via:
- `fade` (Low activity) — conflict wound down without agreement
- `not at risk` — military victory or actor disappearance; signing is no longer possible
- `censored` — conflict still active at data end (Dec 2024), right-censored

In case of coincidence, `first_agreement` dominates: a conflict classified as `first_agreement` retains that label
regardless of its UCDP termination outcome (the agreement is the event, not the termination).

In [149]:
# ── df spell: filter each conflict to [start_date, effective_end_date] ──
# effective_end_date is cause-specific (set by build_termination_outcome):
#   first_agreement → first PA-X or UCDP-agreement month (spell ends at signing)
#   fade / not at risk → UCDP episode end (or last GED month if GED ends before term)
#   censored → last GED month (≤ 2024-12)

_ym    = df_panel['year_mo'].str[:7]
_start = df_panel['start_date'].str[:7]
_end   = df_panel['effective_end_date'].str[:7]

df_spell = df_panel[(_ym >= _start) & (_ym <= _end)].copy()
last_month_per_conflict = df_spell.groupby('conflict_id')['year_mo'].transform('max')
df_spell['is_first_agreement'] = (
    (df_spell['ever_agreement'] == 1) &
    (df_spell['year_mo'] == last_month_per_conflict)
).astype(int)
df_spell['log_conflict_age'] = np.log1p(df_spell['conflict_age'])

## **Quarterly Aggregation**

---

The monthly spell is collapsed to **quarters** (`spell_q`), the unit of analysis, each row is one **conflict × quarter**.

### Aggregation rules

- `is_first_agreement`: `max()` — 1 if any month in the quarter was the agreement event
- `cause_label`: `first()` — constant within a conflict; one outcome per `conflict_id`
- `best`, `n_events`: `sum()` — quarterly totals
- Time-invariant covariates (`experience_total`, incompatibility, etc.): `first()`
- `conflict_age_q` and `log_conflict_age_q` are recomputed as the **quarter count** within
  each conflict's spell (1 = first quarter at risk), replacing the monthly `conflict_age`

### `start_date_q` vs `end_date_q`

- `start_date_q`: first day of the quarter containing `start_date` (spell entry)
- `end_date_q`: first day of the quarter containing `effective_end_date` (cause-specific spell exit — agreement month for signers, UCDP termination or last GED month for non-signers)
- For the hazard model, the unit of time is `conflict_age_q` (quarters since onset), not calendar date

In [150]:
df_spell["yq"]           = pd.to_datetime(df_spell["year_mo"]).dt.to_period("Q")
df_spell['start_date_q'] = pd.to_datetime(df_spell['start_date']).dt.to_period("Q").dt.to_timestamp()
df_spell['end_date_q']   = pd.to_datetime(df_spell['effective_end_date']).dt.to_period("Q").dt.to_timestamp()

_AGG = {
    "best":                        "sum",
    "n_events":                    "sum",
    "is_first_agreement":          "max",
    "ever_agreement":              "max",
    "cause_label":                 "first",
    "agree_timing":                "first",
    "termination_outcome_label":   "first",
    "conflict_age":                "max",
    "isocode":                     "first",
    "country":                     "first",
    "region":                      "first",
    "type_of_conflict":            "first",
    "incompatibility":             "first",
    "territorial_incompatibility": "first",
    "experience_total":            "first",
    "fe_etfra":                    "first",
    "fe_etfra_imp":                "first",
    "fe_cultdiv":                  "first",
    "high_fe_etfra_bin":           "first",
    "pko_onset":                   "first",
    "pko_ever":                    "first",
    "defpact_onset":               "first",
    "defpact_majpow_onset":        "first",
    "loot_onset_strict":           "first",
    "loot_onset":                  "first",
    "bdist_onset":                 "first",
    "capdist_onset":               "first",
    "mountains_onset":             "first",
    "start_date_q":                "first",
    "end_date_q":                  "first",
}

spell_q = (
    df_spell.groupby(["conflict_id", "yq"])
    .agg(_AGG)
    .reset_index()
    .sort_values(["conflict_id", "yq"])
    .reset_index(drop=True)
)

spell_q["conflict_age_q"]     = spell_q.groupby("conflict_id").cumcount() + 1
spell_q["log_conflict_age_q"] = np.log1p(spell_q["conflict_age_q"])
spell_q["high_ia_bin"]        = (spell_q["experience_total"] == 0).astype(int)


In [151]:
_export_cols = [
    'conflict_id', 'yq', 'start_date_q', 'end_date_q',
    # outcome / competing risks
    'is_first_agreement', 'ever_agreement',
    'cause_label', 'termination_outcome_label',
    # baseline hazard
    'conflict_age_q', 'log_conflict_age_q',
    # raw intensity
    'best', 'n_events',
    # information asymmetry (onset)
    'experience_total', 'high_ia_bin',
    # commitment proxies (onset)
    'incompatibility', 'territorial_incompatibility',
    'fe_etfra_imp', 'high_fe_etfra_bin', 'fe_cultdiv',
    # pko_onset: Walter (1997) prior enforcement capacity
    #   → 0 for all 98 pre-1994 conflicts (data gap, not absence)
    'pko_onset',
    # pko_ever: international engagement at any point during spell (endogenous)
    'pko_ever',
    # defpact_onset: any defensive alliance at onset (ATOP via QoG, last updated 2022)
    'defpact_onset',
    # defpact_majpow_onset: defensive alliance with a major power (ATOP 5.1 dyad-year)
    'defpact_majpow_onset',
    # loot_onset_strict: yearly _y only — no future-contamination risk
    'loot_onset_strict',
    # loot_onset: yearly _y + static _s — complete picture (minor caveat for undated deposits)
    'loot_onset',
    # bdist_onset: mean distance to nearest border (km) across onset cells
    'bdist_onset',
    # capdist_onset: mean distance to capital (km) across onset cells
    'capdist_onset',
    # mountains_onset: mean share mountainous terrain across onset cells (PRIO-GRID mountains_mean)
    'mountains_onset',
    # identifiers
    'country', 'region', 'isocode', 'type_of_conflict',
]
_export_cols = list(dict.fromkeys(_export_cols))

_avail = [c for c in _export_cols if c in spell_q.columns]
_miss  = [c for c in _export_cols if c not in spell_q.columns]
if _miss:
    print(f"WARNING — not yet built: {_miss}")

spell_q[_avail].to_csv('../../data/output/conflict_level/spell_q.csv', index=False)
print(f"Saved {len(spell_q[_avail]):,} rows x {len(_avail)} cols → spell_q.csv")
print(f"Columns: {_avail}")

Saved 6,188 rows x 32 cols → spell_q.csv
Columns: ['conflict_id', 'yq', 'start_date_q', 'end_date_q', 'is_first_agreement', 'ever_agreement', 'cause_label', 'termination_outcome_label', 'conflict_age_q', 'log_conflict_age_q', 'best', 'n_events', 'experience_total', 'high_ia_bin', 'incompatibility', 'territorial_incompatibility', 'fe_etfra_imp', 'high_fe_etfra_bin', 'fe_cultdiv', 'pko_onset', 'pko_ever', 'defpact_onset', 'defpact_majpow_onset', 'loot_onset_strict', 'loot_onset', 'bdist_onset', 'capdist_onset', 'mountains_onset', 'country', 'region', 'isocode', 'type_of_conflict']


## **Export**

---

Save the quarterly spell dataset for use in `survival_analysis.ipynb`.
- `spell_q.csv` — loaded by the analysis notebook (Python)
- `spell_q.dta` — for the Stata ClogLog specification (`cloglog_hazard.do`)